<a href="https://colab.research.google.com/github/paplpap/zero-to-AI/blob/main/openai_rag_ex.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 시스템 프롬프트 간단예시

코드 구조

from openai import OpenAI: OpenAI라는 라이브러리에서 핵심 부품(OpenAI 클래스)을 꺼내오는 과정입니다.

client = OpenAI(...): 내 전용 AI 비서를 생성하는 단계입니다. 여러분이 발급받은 API 키가 바로 비서의 '출입증' 역할을 합니다.

messages=[...]: AI와 대화하는 '맥락'을 설정합니다.

role: system: AI에게 **"당신은 어떤 사람인가"**라는 정체성(System Prompt)을 부여합니다. 오늘 우리는 '마케팅 전문가'라는 가면을 씌워줄 것입니다.

role: user: 우리가 실제로 궁금한 질문을 던지는 부분입니다.

response.choices[0]...: AI가 내놓은 여러 답변 중 가장 적절한 첫 번째 결과물만 골라 화면에 보여달라는 명령어입니다.

In [ ]:
from openai import OpenAI

# 1. 내 비서(클라이언트) 소환하기
# 'YOUR_API_KEY' 부분에 발급받은 키를 넣으세요.
client = OpenAI(api_key="YOUR_API_KEY")

# 2. 질문 던지기 (채팅 형식)
response = client.chat.completions.create(
    model="gpt-3.5-turbo",
    messages=[
        {"role": "system", "content": "당신은 유능한 마케팅 전문가입니다."},
        {"role": "user", "content": "신규 런칭하는 '저당 단백질 쉐이크'의 블로그 제목 5개만 뽑아줘."}
    ]
)

print(response.choices[0].message.content)

1. "품질과 효능 모두 갖춘 '저당 단백질 쉐이크'가 올해 런칭!"
2. "'저당 단백질 쉐이크'로 건강과 풍성함을 한 번에 챙기세요!"
3. "충격적인 맛과 영양이 만나는 '저당 단백질 쉐이크'!"
4. "다이어트와 영양을 한 번에! 새로운 '저당 단백질 쉐이크'의 매력"
5. "맛과 효능, 모두 갖춘 '저당 단백질 쉐이크'의 매력에 빠져보세요!"


##나만의 '지식 창고' (Vector DB) 구축하기
단순히 AI와 대화하는 것을 넘어, 이제 AI가 내가 준 문서(PDF, 텍스트 등)를 공부하고 대답하게 만드는 마법을 부려보겠습니다. 이것을 전문 용어로 **RAG(Retrieval-Augmented Generation)**라고 부릅니다.

# 1. 왜 '지식 창고'가 필요할까?
일반적인 GPT는 2023년 혹은 그 이전의 데이터까지만 알고 있습니다. 하지만 우리 회사의 어제자 공지사항이나 나만의 비공개 노트를 물어보면 대답하지 못하죠.

해결책: > 1. 내 데이터를 잘게 쪼개서 **'디지털 도서관(Vector DB)'**에 넣습니다.
2. 질문이 들어오면 도서관에서 관련 내용을 찾아(Retrieve) 옵니다.
3. 찾은 내용을 GPT에게 전달하며 **"이걸 참고해서 답변해줘!"**라고 시킵니다.


# 2. 코드 구조 파헤치기 (준비 단계)
OpenAIEmbeddings(): 텍스트를 AI가 계산할 수 있는 숫자 형태(벡터)로 변환하는 **'AI용 번역기'**입니다.

Document(...): 평범한 텍스트 데이터를 AI가 관리하기 좋은 '문서 객체' 형태로 포장하는 과정입니다.

Chroma.from_documents(...): 숫자로 변환된 지식들을 체계적으로 저장하고 검색할 수 있는 **'디지털 지식 창고(Vector DB)'**를 생성합니다.

os.environ[...]: 내 소중한 API 키를 시스템 환경 변수에 등록하여 안전하게 사용할 수 있도록 설정합니다.

In [ ]:
!pip install -qU langchain langchain-openai langchain-community chromadb langchain-core

In [ ]:
import os
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import Chroma
from langchain_core.documents import Document
from langchain_classic.chains.retrieval_qa.base import RetrievalQA

In [ ]:
# 1. API 키 설정
os.environ["OPENAI_API_KEY"] = openai_key

In [ ]:
# 2. 데이터 준비 (Vector DB 생성)
my_knowledge = [
    "우리 회사의 여름 휴가는 총 5일이며, 7월에서 8월 사이에 사용해야 합니다.",
    "업무용 도서 구입비는 월 5만 원까지 지원됩니다."
]

documents = [Document(page_content=text) for text in my_knowledge]
vector_db = Chroma.from_documents(
    documents,
    OpenAIEmbeddings(),
    collection_name="my_local_db"
)

In [ ]:
# 3. 모델 및 RAG 체인 설정
chat_model = ChatOpenAI(model="gpt-4o", temperature=0)

qa_chain = RetrievalQA.from_chain_type(
    llm=chat_model,
    chain_type="stuff",
    retriever=vector_db.as_retriever()
)

In [ ]:
# 4. 실행
question = "니가 알고 있는 우리 회사 정보를 다 말해봐"
response = qa_chain.invoke(question)
print(response['result'])

죄송하지만, 제공된 정보 외에는 추가적인 회사 정보를 가지고 있지 않습니다. 알려진 정보로는 여름 휴가가 총 5일이며, 7월에서 8월 사이에 사용해야 한다는 것과 업무용 도서 구입비가 월 5만 원까지 지원된다는 것입니다. 추가적인 정보가 필요하시면 회사의 공식 자료나 담당 부서에 문의하시기 바랍니다.
